In [1]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [10]:
import numpy as np 
from fiftyone import ViewField

In [3]:
import sys
%load_ext autoreload
%autoreload 2

In [4]:
fo.list_datasets()

['FLPLAN',
 'flplan200',
 'flplanpatches',
 'nc765',
 'nc765_T1024ov224',
 'tiles_merged',
 'umflplan_T1024ov224',
 'wp125',
 'wp125_T1024ov224']

In [5]:
dataset = fo.load_dataset('tiles_merged')


In [6]:
# load embeddings 
from sklearn.preprocessing import normalize

data_embedding_tile        = np.load("/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/chapter3/embeddings_dinov3/merged_1024o224_fullimage_embeddings.npz")

sample_ids = data_embedding_tile['sample_ids']

embedding_retrieved = data_embedding_tile['embeddings']

In [7]:
label_lookup = {
        s.id: s.get_field("type_label")
        for s in dataset.select_fields("type_label")
    }

In [8]:
dataset.count_values("dataset_sr")

{'UM_flplan': 2755, 'NC_flplan': 5728, 'WPmix': 2640}

In [9]:
dataset

Name:        tiles_merged
Media type:  image
Num samples: 11123
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    type_label:       fiftyone.core.fields.StringField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    region:           fiftyone.core.fields.StringField
    mission_name:     fiftyone.core.fields.StringField
    parent_name:      fiftyone.core.fields.StringField
    gt_field:         fiftyone.core.fields.StringField
    min_area_ratio:   fiftyone.core.fields.FloatField
    n_boxes_in_tile:  fiftyone.core.fields.I

In [64]:
dataset.count_values("mission_name")

{'GAM': 405,
 'Flight_198': 584,
 'Flight_209': 664,
 'Flight_214': 152,
 'FRIWEN': 945,
 'Flight_234': 16,
 'Flight_199': 64,
 'Flight_235': 32,
 'Flight_215': 296,
 'UM': 3940,
 'Flight_226': 40,
 'Flight_219': 184,
 'Flight_197': 400,
 'Flight_205': 72,
 'Flight_211': 136,
 'Flight_218': 176,
 'Flight_196': 504,
 'Flight_220': 160,
 'Flight_207': 1024,
 'Flight_231': 96,
 'Flight_200': 80,
 'Flight_222': 136,
 'Flight_195': 48,
 'Flight_213': 328,
 'Flight_225': 16,
 'MANTASANDY': 105,
 'Flight_232': 184,
 'Flight_233': 48,
 'Flight_212': 288}

In [64]:
# WITHOUT -UM  - train and test set 
# wp_mix = dataset.match(
#     (ViewField("dataset_sr").contains_str("WPmix"))
#     & (ViewField("mission_name").is_in(("MANTASANDY", "FRIWEN", "GAM")))
# )

# WITH UM
wp_mix = dataset.match(
    (ViewField("dataset_sr").contains_str("WPmix")))

nc_view = dataset.match(ViewField("dataset_sr").contains_str("NC_flplan"))

train_set_view = wp_mix.concat(nc_view)
train_set_id   = train_set_view.values('id')

um_test_ids = dataset.match(ViewField("dataset_sr").contains_str("UM_flplan")).values('id')


In [62]:
train_set_view.count_values('dataset_sr')

{'NC_flplan': 5728, 'WPmix': 2640}

In [65]:
train_set_view.count_values('mission_name')

{'Flight_197': 400,
 'Flight_235': 32,
 'Flight_196': 504,
 'Flight_234': 16,
 'Flight_215': 296,
 'Flight_232': 184,
 'Flight_213': 328,
 'Flight_205': 72,
 'Flight_198': 584,
 'Flight_231': 96,
 'Flight_207': 1024,
 'Flight_214': 152,
 'Flight_225': 16,
 'Flight_226': 40,
 'GAM': 405,
 'MANTASANDY': 105,
 'Flight_209': 664,
 'Flight_220': 160,
 'UM': 1185,
 'Flight_222': 136,
 'Flight_219': 184,
 'Flight_212': 288,
 'FRIWEN': 945,
 'Flight_200': 80,
 'Flight_218': 176,
 'Flight_199': 64,
 'Flight_233': 48,
 'Flight_211': 136,
 'Flight_195': 48}

In [66]:
#Lookup table id -> positive/negative 
label_lookup = {
    s.id: s.get_field("type_label")
    for s in dataset.select_fields("type_label")
}

In [67]:
# map in order to not mixed embedding and index: sample_id -> row index in the embedding array 
sample_id_to_row = {sid: i for i, sid in enumerate(sample_ids)}

def _build_X_y(ids):
    """
    Builds X and y from the SAME id list, in the SAME order, in one pass.
    This guarantees row i of X and row i of y always refer to the same
    sample.
    """
    rows, labels, missing_embedding, missing_label = [], [], [], []

    for sid in ids:
        row_idx = sample_id_to_row.get(sid)
        if row_idx is None:
            missing_embedding.append(sid)
            continue

        lab = label_lookup.get(sid)
        if lab == "positive":
            y_val = 1
        elif lab == "negative":
            y_val = 0
        else:
            missing_label.append(sid)
            continue

        rows.append(row_idx)
        labels.append(y_val)

    X = embedding_retrieved[rows]
    y = np.array(labels)
    kept_ids = np.array([ids[i] for i in range(len(ids))
                         if ids[i] not in missing_embedding and ids[i] not in missing_label])
    return X, y, kept_ids, missing_embedding, missing_label


X_train, y_train, train_ids_kept, miss_emb_tr, miss_lab_tr = _build_X_y(train_set_id)
X_test,  y_test,  test_ids_kept,  miss_emb_te, miss_lab_te = _build_X_y(um_test_ids)

print(f"Train: {X_train.shape[0]} rows  (missing_embedding={len(miss_emb_tr)}, "
      f"missing_label={len(miss_lab_tr)})")
print(f"Test:  {X_test.shape[0]} rows  (missing_embedding={len(miss_emb_te)}, "
      f"missing_label={len(miss_lab_te)})")

X_train = normalize(X_train, norm="l2")
X_test  = normalize(X_test,  norm="l2")

Train: 8368 rows  (missing_embedding=0, missing_label=0)
Test:  2755 rows  (missing_embedding=0, missing_label=0)


In [68]:
X_train.shape

(8368, 1024)

In [69]:
X_test.shape

(2755, 1024)

In [70]:
y_train.shape

(8368,)

In [71]:
y_test.shape

(2755,)

In [37]:
from src.classification import fit_logistic_regression, evaluate_predictions, fit_random_forest

In [72]:
result_lr  = fit_logistic_regression(
    X_train, y_train, X_test, y_test, test_ids=um_test_ids
)


=== Logistic Regression ===
  Accuracy : 0.9829
  Precision: 0.8133
  Recall   : 0.8652
  F1       : 0.8385
  ROC-AUC  : 0.9888
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2586   28]
 [  19  122]]
              precision    recall  f1-score   support

    negative       0.99      0.99      0.99      2614
    positive       0.81      0.87      0.84       141

    accuracy                           0.98      2755
   macro avg       0.90      0.93      0.91      2755
weighted avg       0.98      0.98      0.98      2755

  Failed predictions: 47 / 2755


In [14]:
result_lr.keys()

dict_keys(['model', 'y_pred', 'y_proba', 'metrics', 'results_df', 'failed_ids'])

In [15]:
result_lr['results_df']

,sample_id,y_true,y_pred,correct,model,y_proba
0,6a33d1a619d8f6f6ef5692ea,1,0,False,Logistic Regression,0.204260
1,6a33d1a619d8f6f6ef5692eb,1,0,False,Logistic Regression,0.082985
2,6a33d1a619d8f6f6ef5692ec,1,1,True,Logistic Regression,0.625178
3,6a33d1a619d8f6f6ef5692ed,1,1,True,Logistic Regression,0.956245
4,6a33d1a619d8f6f6ef5692ee,1,1,True,Logistic Regression,0.960776
...,...,...,...,...,...,...
2750,6a33d1a719d8f6f6ef569da8,0,0,True,Logistic Regression,0.379156
2751,6a33d1a719d8f6f6ef569da9,0,0,True,Logistic Regression,0.191679
2752,6a33d1a719d8f6f6ef569daa,0,0,True,Logistic Regression,0.224964
2753,6a33d1a719d8f6f6ef569dab,0,0,True,Logistic Regression,0.198591


In [16]:
%reload_ext
from src import classification

UsageError: Missing module name.


In [17]:
from src.classification import fit_logistic_regression, fit_random_forest, fit_mlp

In [73]:
from src.classification import fit_logistic_regression, fit_random_forest

#result_lr = fit_logistic_regression(X_train, y_train, X_test, y_test, test_ids=um_test_ids)
result_rf = fit_random_forest(X_train, y_train, X_test, y_test, test_ids=um_test_ids)



=== Random Forest ===
  Accuracy : 0.9866
  Precision: 0.9483
  Recall   : 0.7801
  F1       : 0.8560
  ROC-AUC  : 0.9885
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2608    6]
 [  31  110]]
              precision    recall  f1-score   support

    negative       0.99      1.00      0.99      2614
    positive       0.95      0.78      0.86       141

    accuracy                           0.99      2755
   macro avg       0.97      0.89      0.92      2755
weighted avg       0.99      0.99      0.99      2755

  Failed predictions: 37 / 2755


In [74]:
result_rf['results_df'].loc[(result_rf['results_df']["y_true"])==1&(result_rf['results_df']["y_pred"]==0)]

,sample_id,y_true,y_pred,correct,model,y_proba
0,6a33d1a619d8f6f6ef5692ea,1,0,False,Random Forest,0.230000
1,6a33d1a619d8f6f6ef5692eb,1,0,False,Random Forest,0.023333
9,6a33d1a619d8f6f6ef5692f3,1,0,False,Random Forest,0.296667
10,6a33d1a619d8f6f6ef5692f4,1,0,False,Random Forest,0.303333
14,6a33d1a619d8f6f6ef5692f8,1,0,False,Random Forest,0.246667
17,6a33d1a619d8f6f6ef5692fb,1,0,False,Random Forest,0.236667
31,6a33d1a619d8f6f6ef569309,1,0,False,Random Forest,0.486667
32,6a33d1a619d8f6f6ef56930a,1,0,False,Random Forest,0.426667
33,6a33d1a619d8f6f6ef56930b,1,0,False,Random Forest,0.446667
34,6a33d1a619d8f6f6ef56930c,1,0,False,Random Forest,0.446667


In [75]:
failed_df = result_rf['results_df'].loc[(result_rf['results_df']["y_true"])==1&(result_rf['results_df']["y_pred"]==0)]
failed_df.loc[failed_df['y_true']==1]
list_fn = failed_df.loc[failed_df['y_true']==1]['sample_id'].to_list()
len(list_fn)

31

In [76]:
session = fo.launch_app(dataset,auto=False,port=5151)

Session launched. Run `session.show()` to open the App in a cell output.


In [77]:
view = dataset.select(list_fn)
session.view = view

In [55]:
%reload_ext src

In [79]:
from src.classification import fit_mlp, DugongMLP


# Or swap in a different architecture entirely
result_mlp2 = fit_mlp(
    X_train, y_train, X_test, y_test, test_ids=um_test_ids,
    model_fn=lambda: DugongMLP(input_dim=X_train.shape[1]),  
)


  Train: 7112 (pos=2044)  Val: 1256 (pos=361)  device=cuda
  pos_weight (n_neg/n_pos) = 2.479
  Model: DugongMLP


  epoch   1  train_loss=0.5498  val_loss=0.4422  best=0.4422  no_improve=0
  epoch  10  train_loss=0.3301  val_loss=0.3103  best=0.2917  no_improve=6
  epoch  20  train_loss=0.2490  val_loss=0.2451  best=0.2367  no_improve=5
  epoch  30  train_loss=0.2442  val_loss=0.2313  best=0.2151  no_improve=2
  epoch  40  train_loss=0.1771  val_loss=0.2484  best=0.2021  no_improve=9
  Early stopping at epoch 46 (no val improvement for 15 epochs).
  Restored weights from best val_loss=0.2021
  Tuned threshold (max f1 on val) = 0.590  (val f1=0.9178)

=== MLP ===
  Accuracy : 0.9858
  Precision: 0.8923
  Recall   : 0.8227
  F1       : 0.8561
  ROC-AUC  : 0.9826
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2600   14]
 [  25  116]]
              precision    recall  f1-score   support

    negative       0.99      0.99      0.99      2614
    positive       0.89      0.82      0.86       141

    accuracy                           0.99      2755
   macro avg       0.94      0.91      0.92      2755
weight

In [56]:
import torch
import torch.nn as nn
from src.classification import fit_mlp, DugongMLP

In [80]:

# smaller mlp
class WiderMLP(nn.Module):
    def __init__(self, input_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

result_mlp3 = fit_mlp(
    X_train, y_train, X_test, y_test, test_ids=um_test_ids,
    model_fn=lambda: WiderMLP(input_dim=X_train.shape[1]),
    val_size = 0.10,
    batch_size = 32,
    max_epochs = 100,
    patience = 25,
    lr = 1e-3,
)

  Train: 7531 (pos=2164)  Val: 837 (pos=241)  device=cuda
  pos_weight (n_neg/n_pos) = 2.480
  Model: WiderMLP
  epoch   1  train_loss=0.6179  val_loss=0.4633  best=0.4633  no_improve=0
  epoch  10  train_loss=0.3138  val_loss=0.3148  best=0.3148  no_improve=0
  epoch  20  train_loss=0.2866  val_loss=0.3386  best=0.2814  no_improve=2
  epoch  30  train_loss=0.2667  val_loss=0.3253  best=0.2713  no_improve=6
  epoch  40  train_loss=0.2622  val_loss=0.2728  best=0.2593  no_improve=9
  epoch  50  train_loss=0.2539  val_loss=0.2698  best=0.2565  no_improve=4
  epoch  60  train_loss=0.2578  val_loss=0.3692  best=0.2565  no_improve=14
  epoch  70  train_loss=0.2492  val_loss=0.2599  best=0.2565  no_improve=24
  Early stopping at epoch 71 (no val improvement for 25 epochs).
  Restored weights from best val_loss=0.2565
  Tuned threshold (max f1 on val) = 0.395  (val f1=0.8953)

=== MLP ===
  Accuracy : 0.9746
  Precision: 0.6983
  Recall   : 0.8865
  F1       : 0.7812
  ROC-AUC  : 0.9868
  Con

In [81]:
## save df 
import pandas as pd 
final_df = pd.concat([
    result_mlp2['results_df'],
    result_rf['results_df'],
    result_lr['results_df']
])

final_df['DOMAIN_SHIFT'] = "NCandWPmixALL"
out_folder = "/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/chapter3/results_classification"
out_nme = "nctrained_WPmixALL_UMtest.parquet"
final_df.to_parquet(os.path.join(out_folder,out_nme))

In [82]:
final_df

,sample_id,y_true,y_pred,correct,model,y_proba,DOMAIN_SHIFT
0,6a33d1a619d8f6f6ef5692ea,1,0,False,MLP,0.003480,NCandWPmixALL
1,6a33d1a619d8f6f6ef5692eb,1,0,False,MLP,0.002633,NCandWPmixALL
2,6a33d1a619d8f6f6ef5692ec,1,1,True,MLP,0.909066,NCandWPmixALL
3,6a33d1a619d8f6f6ef5692ed,1,1,True,MLP,0.992924,NCandWPmixALL
4,6a33d1a619d8f6f6ef5692ee,1,1,True,MLP,0.999224,NCandWPmixALL
...,...,...,...,...,...,...,...
2750,6a33d1a719d8f6f6ef569da8,0,0,True,Logistic Regression,0.195740,NCandWPmixALL
2751,6a33d1a719d8f6f6ef569da9,0,0,True,Logistic Regression,0.118306,NCandWPmixALL
2752,6a33d1a719d8f6f6ef569daa,0,0,True,Logistic Regression,0.142134,NCandWPmixALL
2753,6a33d1a719d8f6f6ef569dab,0,0,True,Logistic Regression,0.137941,NCandWPmixALL


# Create a GIF

In [45]:
%reload_ext src

In [49]:
from src.gif_plot import create_failure_reveal_gif

info = create_failure_reveal_gif(
    filepaths= filepath_fn_rf,   # your list of image paths where the model missed
    output_path="NCtrained_RF_dugong_failures.gif",
    raw_duration_s=2,
    reveal_duration_s=1,
)

GIF saved -> NCtrained_RF_dugong_failures.gif
  Tiles: 20  Frames: 40 (2 per tile)
  Size: 19095.9 KB
  Requested durations (ms), first 4 frames: [2000, 1000, 2000, 1000]
  Actual durations read back from file, first 4 frames: [2000, 1000, 2000, 1000]
